# Fine-tune Qwen2.5-VL-7B for handwritten notes → Markdown (local GPU)

**Task.** Input: a preprocessed page with Surya's layout boxes drawn on it, plus that page's layout tags as JSON. Target: the page's Markdown file in `data/annotations/`.

**Method.** 4-bit QLoRA on `Qwen/Qwen2.5-VL-7B-Instruct` (Hugging Face weights; the Ollama `qwen2.5vl:7b` copy is a GGUF file for inference and cannot be trained), bf16 on an RTX 5060 Ti (16 GB). Plain `transformers` + `peft`. The loss covers only the answer: the prompt, image and layout JSON are masked.

**Run order.** 0 → 1 → 2 → 3 → 4 → 5 (baseline) → 6 (pilot, `PILOT = True`) → restart kernel → 0 → 1 → 2 → 3 → 4 → 6 (`PILOT = False`) → 7 → 8. Section 9 is optional.

**Folder this notebook expects** (copy the whole repo to the GPU machine):
```
ie643_project/
  00_dataset/ 03_recognition/ 05_evaluation/       code (from git)
  data/annotations/*.md   data/splits.json   data/*.jsonl          (from git)
  data/tags_draft/*.json   data/boxed/*.jsonl                   Surya layout tags, all 410 pages (from git)
  data/processed/*.jpg            corrected preprocessed pages      (rebuilt on this machine, section 0)
  data/processed_boxed/*.jpg      same pages with Surya boxes        (rebuilt on this machine, section 0)
```
Open this notebook from `ie643_project/03_recognition/`.

## 0. One-time setup (in a terminal, not in the notebook)

The `.qwen` venv has no training libraries yet. The RTX 5060 Ti (Blackwell) needs a PyTorch build for CUDA 12.8 or newer.
```bash
source .qwen/bin/activate
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
pip install -r ie643_project/03_recognition/requirements-train.txt
python -m ipykernel install --user --name qwen-ft --display-name "qwen-ft (.qwen)"
```
Then pick the **qwen-ft (.qwen)** kernel for this notebook.

**Rebuild the images on this machine** (about 5-10 minutes on CPU; no download needed). The raw scans, the fixed preprocessing code and Surya's tags are all in the repo:
```bash
cd ie643_project
pip install opencv-python-headless numpy
python3 01_preprocessing/batch_preprocess.py --workers 8 --force   # -> data/processed/ (must say 55 page-warped)
python3 02_layout/draw_boxes.py                                    # -> data/processed_boxed/ (must say 410 with boxes)
python3 00_dataset/check_dataset.py                                # must end with 0 FAIL
```
`draw_boxes.py` refuses to draw on a page whose size does not match its tags, so a wrong image version shows up here.

**Free the GPU from Ollama** before training (a loaded Ollama model holds several GB of VRAM):
```bash
ollama ps                    # what is loaded
ollama stop qwen2.5vl:7b     # unload it (or: sudo systemctl stop ollama)
```
The first model load downloads about 16 GB to `~/.cache/huggingface`.

**Remote machine?** The same steps exist as a script for `tmux`: `python3 03_recognition/finetune_qwen25vl.py --pilot`, then without `--pilot`. It survives a dropped connection, which a notebook does not.

## 1. Settings

In [ ]:
VARIANT    = "boxed_json"   # "boxed_json" (boxes + layout JSON), "boxed" (boxes only), "clean" (no layout)
MAX_PIXELS = 1_000_000      # page is shrunk to <= this many pixels (~1250 image tokens); lower it if you run out of memory
PILOT      = True           # True: 5 training steps to measure memory and speed. False: the real run.
EPOCHS     = 3
LR         = 1e-4
GRAD_ACCUM = 8              # batch size is 1; 8 pages per optimizer step
LORA_R     = 16
MAX_NEW_TOKENS = 1280       # longest label is ~830 tokens
BASELINE_PAGES = 10         # val pages for the zero-shot baseline (25 = all)

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
OUT_DIR  = f"runs/qwen25vl_{VARIANT}"

## 2. Environment check

In [ ]:
import os, sys, subprocess, torch, transformers, peft
here = os.getcwd()  # the notebook's folder
ROOT = here if os.path.isdir(os.path.join(here, "00_dataset")) else os.path.dirname(here)
assert os.path.isdir(os.path.join(ROOT, "00_dataset")), "open this notebook from ie643_project/03_recognition/"
sys.path.insert(0, os.path.join(ROOT, "03_recognition"))
print("repo root  :", ROOT)
print("torch      :", torch.__version__, "| CUDA", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
print("transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "No GPU visible: check the PyTorch CUDA build (section 0)"
print("GPU        :", torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0),
      "| bf16:", torch.cuda.is_bf16_supported())
free, total = torch.cuda.mem_get_info()
print(f"VRAM free  : {free/2**30:.1f} of {total/2**30:.1f} GB")
if free / total < 0.85:
    print("WARNING: something else holds GPU memory (Ollama?). Run `ollama stop qwen2.5vl:7b` and re-run this cell.")
print(subprocess.run("nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout or "no other GPU processes")

## 3. Data check
Stops here if anything is missing or out of date. Fix it before training.

In [ ]:
import csv, json, collections
import qwen_data as Q
D = os.path.join(ROOT, "data")

rows = list(csv.DictReader(open(os.path.join(D, "processed", "preprocess_report.csv"))))
warped = sum(r["crop"] == "quad" for r in rows)
print("preprocessed pages:", len(rows), "| warped:", warped, "(55 = corrected images, 62 = old)")
assert warped == 55, "data/processed is not the corrected version: copy the corrected images.zip"

problems = Q.check_inputs(VARIANT)
print("input problems for", VARIANT, ":", len(problems))
for p in problems[:10]: print("  ", p)
assert not problems, "fix the problems above (missing images/tags, or tags from old images)"

for split in ("train", "val", "test"):
    ids = Q.page_ids(split)
    print(f"{split:5s} {len(ids):3d} pages", dict(collections.Counter(i.split('_')[0] for i in ids)))

## 4. Load processor, build examples, look at one

In [ ]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=4*28*28, max_pixels=MAX_PIXELS)
collate = Q.make_collate_fn(processor)

train_ds = [Q.build_example(p, VARIANT, MAX_PIXELS) for p in Q.page_ids("train")]
val_ds   = [Q.build_example(p, VARIANT, MAX_PIXELS) for p in Q.page_ids("val")]
print(len(train_ds), "train,", len(val_ds), "val")

lengths = []
for ex in train_ds:
    b = collate([ex]); lengths.append((b["input_ids"].shape[1], int((b["labels"] != -100).sum())))
print("tokens per example: max", max(l[0] for l in lengths), "| answer tokens max", max(l[1] for l in lengths))

ex = next(e for e in train_ds if e["id"] == "mlt_week03_page001") if any(e["id"] == "mlt_week03_page001" for e in train_ds) else train_ds[0]
print("\nexample:", ex["id"], "image fed:", ex["images"][0].size)
print(ex["messages"][0]["content"][1]["text"][:700], "...")
b = collate([ex])
print("\nlearned part (start):", repr(processor.tokenizer.decode(b["input_ids"][0][b["labels"][0] != -100])[:150]))
from IPython.display import display
im = ex["images"][0].copy(); im.thumbnail((700, 700)); display(im)

## 5. Load the model (4-bit) and score the zero-shot baseline on validation
This is the number fine-tuning has to beat. Predictions go to `preds/`, scores come from `05_evaluation/evaluate.py` (CER, WER, text-only CER, equation exact match; lower CER/WER is better).

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.bfloat16)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa")
print(f"model loaded, GPU memory {torch.cuda.memory_allocated()/2**30:.1f} GB")

In [ ]:
import time
def predict(model, pages, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    model.eval()
    for i, p in enumerate(pages, 1):
        path = os.path.join(out_dir, p + ".md")
        if os.path.exists(path): continue
        ex = Q.build_example(p, VARIANT, MAX_PIXELS, with_answer=False)
        text = processor.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[ex["images"]], return_tensors="pt").to(model.device)
        t = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.05, use_cache=True)
        pred = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        open(path, "w").write(pred)
        print(f"[{i}/{len(pages)}] {p}: {len(pred)} chars, {time.time()-t:.0f}s")

def score(pred_dir):
    r = subprocess.run([sys.executable, os.path.join(ROOT, "05_evaluation", "evaluate.py"),
                        "--pred", pred_dir, "--split", os.path.join(D, "val.jsonl")],
                       cwd=ROOT, capture_output=True, text=True)
    print(r.stdout or r.stderr)

BASE_PRED = os.path.join(ROOT, "preds", f"base_{VARIANT}_val")
predict(model, Q.page_ids("val")[:BASELINE_PAGES], BASE_PRED)
score(BASE_PRED)

## 6. Fine-tune (QLoRA)
With `PILOT = True` this runs 5 steps (40 pages) and reports memory and speed. Then set `PILOT = False`, **restart the kernel**, and run 1, 2, 3, 4, both cells of 5 (the baseline predictions are cached in `preds/`, so this is quick) and 6 again.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

# Gradient checkpointing + input gradients, kept in bf16. (prepare_model_for_kbit_training would
# upcast the embedding and output layers to fp32: ~2 GB of extra VRAM for no benefit in bf16.)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False
lora = LoraConfig(r=LORA_R, lora_alpha=2*LORA_R, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    max_steps=5 if PILOT else -1,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.03, max_grad_norm=1.0,
    bf16=True, optim="paged_adamw_8bit",
    logging_steps=1 if PILOT else 5,
    eval_strategy="no" if PILOT else "epoch",
    save_strategy="no" if PILOT else "epoch", save_total_limit=2,
    load_best_model_at_end=not PILOT, metric_for_best_model="eval_loss", greater_is_better=False,
    remove_unused_columns=False, dataloader_num_workers=0, report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)

torch.cuda.reset_peak_memory_stats(); t0 = time.time()
result = trainer.train()
secs = (time.time() - t0) / max(result.global_step, 1)
print(f"\n{result.global_step} steps, {secs:.0f} s/step ({GRAD_ACCUM} pages), peak GPU {torch.cuda.max_memory_allocated()/2**30:.1f} GB")
if PILOT:
    steps = -(-len(train_ds) // GRAD_ACCUM) * EPOCHS
    print(f"full run: {steps} steps -> about {steps*secs/3600:.1f} h (+ ~{len(val_ds)*EPOCHS} val pages of evaluation)")

## 7. Save the adapter (real run only)

In [ ]:
assert not PILOT, "this was a pilot; set PILOT = False and retrain"
ADAPTER = os.path.join(OUT_DIR, "final_adapter")
trainer.save_model(ADAPTER); processor.save_pretrained(ADAPTER)
print("saved", ADAPTER); print(trainer.state.log_history[-3:])

## 8. Score the fine-tuned model on validation (same pages as the baseline)

In [ ]:
model.config.use_cache = True
FT_PRED = os.path.join(ROOT, "preds", f"lora_{VARIANT}_val")
predict(trainer.model, Q.page_ids("val")[:BASELINE_PAGES], FT_PRED)
print("BASELINE (zero-shot)"); score(BASE_PRED)
print("FINE-TUNED"); score(FT_PRED)

To compare variants, rerun the notebook with `VARIANT = "clean"` (and/or `"boxed"`); the validation pages and settings stay the same. Keep the **test split** untouched until its labels are checked by hand (`data/VERIFICATION.md`).

## 9. (Optional, not tested) Merge the adapter for deployment

Needs about 32 GB of system RAM (the merge runs on the CPU in bf16). The result is a normal Hugging Face model folder. Serving it with `transformers` (as in `predict()` above) is the tested route. Importing it into Ollama requires converting a Qwen2.5-VL model with its vision encoder to GGUF; check that your Ollama / llama.cpp version supports that for Qwen2.5-VL before relying on it.

In [ ]:
RUN_MERGE = False
if RUN_MERGE:
    from peft import PeftModel
    base = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu")
    merged = PeftModel.from_pretrained(base, os.path.join(OUT_DIR, "final_adapter")).merge_and_unload()
    merged.save_pretrained(os.path.join(OUT_DIR, "merged"), safe_serialization=True)
    processor.save_pretrained(os.path.join(OUT_DIR, "merged"))
    print("merged model in", os.path.join(OUT_DIR, "merged"))